# 02 — Tratamento dos Dados

## 1. Objetivo do Notebook

> Este notebook tem como objetivo aplicar as decisões de tratamento justificadas na etapa de exploração dos dados.

> A partir da análise exploratória, foram definidas as seguintes decisões:

- remover o mês incompleto de outubro de 2023;
- remover apenas registros com possível erro real de PDV, onde o desconto é maior que o valor bruto em vendas positivas;
- manter registros negativos, pois representam possíveis devoluções, estornos ou ajustes financeiros;
- remover a coluna `valor_troca`, pois foi identificada como zerada em toda a base;
- preencher valores nulos em `codigo_produto` com uma categoria específica;
- criar flags para registros negativos e vendas de alto valor;
- gerar uma base transacional tratada;
- agregar os dados em granularidade mensal por loja;
- salvar a base mensal que será utilizada no notebook de modelagem.

> A variável principal de faturamento será mantida como `faturamento_liquido`, pois representa o valor líquido efetivamente registrado na base.

## 2. Importação das bibliotecas

In [1]:
import os 
import glob 
import warnings

import numpy as np 
import pandas as pd 

warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

## 3. Leitura dos Dados Brutos

> Nesta etapa, os arquivos brutos mensais são lidos e consolidados em um único DataFrame.

> O tratamento será feito a partir da base original, e não a partir do notebook de exploração, para garantir que este notebook seja independente e reprodutível.

In [2]:
# Definindo o caminho dos arquivos brutos 
path = '../data/raw/CASTING_*.csv'

# Buscando todos os arquivos CSV dentro da pasta raw  
path_csv = glob.glob(path)

print('Arquivos encontrados:', {len(path_csv)})

# Loop para mostrar os arquivos 
for arquivo in sorted(path_csv):
    print(arquivo)

Arquivos encontrados: {8}
../data/raw\CASTING_DB063.csv
../data/raw\CASTING_DB069.csv
../data/raw\CASTING_DB088.csv
../data/raw\CASTING_DB092.csv
../data/raw\CASTING_DB101.csv
../data/raw\CASTING_DB102.csv
../data/raw\CASTING_DB109.csv
../data/raw\CASTING_DB113.csv


In [3]:
# Lendo todos or arquivos utilizando uma list compreension
dfs = [pd.read_csv(arquivo, low_memory= False) for arquivo in path_csv]

In [6]:
# Concatedando os df em um único 
df = pd.concat(dfs, ignore_index = True)

print(f'Dimensão da base bruta: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

dicionario_colunas = {
    'VUF_VLRLIQFINAL': 'faturamento_liquido',
    'VUF_VLRBRUTOVENDA': 'valor_bruto_venda',
    'VUF_VLRDESCONTO': 'valor_desconto',
    'VUF_VLRTROCA': 'valor_troca',
    'VUF_CODIGO': 'id_transacao',
    'CLV_BANCO': 'loja',
    'VUF_CODIGO_BOLETO': 'codigo_boleto',
    'UND_CODIGO': 'codigo_unidade',
    'FUN_CODIGO': 'codigo_funcionario',
    'VUF_DT': 'data_venda',
    'PRO_CODIGO': 'codigo_produto',
    'CAT_CODIGO': 'codigo_categoria',
    'VUF_QTBOLETO': 'qtd_boletos',
    'VUF_QTPRODUTO': 'qtd_produtos'
}

df = df.rename(columns=dicionario_colunas)

Dimensão da base bruta: 8,297,456 linhas x 14 colunas


In [7]:
# Conversao inicial da coluna de data 
df['data_venda'] = pd.to_datetime(df['data_venda'], errors = 'coerce')

print(f'Data minima:', {df['data_venda'].min()})
print(f'Data máxima:', {df['data_venda'].max()})

Data minima: {Timestamp('2022-01-01 00:00:00')}
Data máxima: {Timestamp('2023-10-01 00:00:00')}


## 4. Função de Tratamento Transacional

> A função abaixo aplica as regras de tratamento definidas a partir da EDA.

> A ideia é transformar a base bruta em uma base transacional tratada, mantendo os registros relevantes para o faturamento e criando flags que ajudem a rastrear comportamentos importantes, como registros negativos e vendas de alto valor.

> Importante: registros negativos não serão removidos, pois impactam diretamente o `faturamento_liquido` real.

In [8]:
def tratamento_dados_transacionais(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica o pipeline de tratamento da base transacional.

    Etapas:
    1. Copia a base original para evitar alterações no DataFrame original.
    2. Converte a coluna de data para datetime.
    3. Remove outubro de 2023, identificado como mês incompleto na EDA.
    4. Remove possíveis erros de PDV (desconto maior que o valor bruto em vendas positivas).
    5. Preenche valores nulos em codigo_produto.
    6. Remove a coluna valor_troca caso esteja zerada em toda a base.
    7. Cria uma flag de inconsistência financeira.
    8. Cria uma flag para devoluções/estornos.
    9. Cria uma flag para vendas de alto valor por loja (Percentil 99).
    10. Cria atributos temporais.
    11. Ordena a base final.
    """


    # Evita alterar o DataFrame original
    df = df.copy()

    print(f"[IN] Dimensão original: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

    
    # 1. Garantir que a coluna de data esteja no formato datetime
    
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")

    
    # 2. Remover registros sem data
    
    n_datas_nulas = df["data_venda"].isna().sum()

    if n_datas_nulas > 0:
        df = df[df["data_venda"].notna()]
        print(f"Datas nulas removidas: {n_datas_nulas:,}")
    else:
        print("Datas nulas removidas: 0")

    
    # 3. Remover outubro/2023 (mês incompleto)
    
    n_outubro = (df["data_venda"] >= "2023-10-01").sum()

    df = df[df["data_venda"] < "2023-10-01"]

    print(f"Outubro/2023 removido: {n_outubro:,} linhas")

    
    # 4. Remover possíveis erros de PDV
    
    mask_erro_pdv = (
        (df["valor_desconto"] > df["valor_bruto_venda"])
        & (df["valor_bruto_venda"] > 0)
    )

    n_erro_pdv = mask_erro_pdv.sum()

    df = df[~mask_erro_pdv]

    print(f"Possíveis erros de PDV removidos: {n_erro_pdv:,}")

    
    # 5. Tratar valores nulos em codigo_produto
    
    n_codigo_produto_nulo = df["codigo_produto"].isna().sum()

    df["codigo_produto"] = df["codigo_produto"].fillna("PRODUTO_NAO_MAPEADO")

    print(f"Código de produto preenchido: {n_codigo_produto_nulo:,} registros")

    
    # 6. Remover coluna valor_troca caso esteja zerada
    
    if "valor_troca" in df.columns:

        n_troca_nao_zero = (df["valor_troca"] != 0).sum()

        if n_troca_nao_zero == 0:
            df = df.drop(columns=["valor_troca"])
            print("Coluna valor_troca removida: todos os valores eram iguais a zero.")

        else:
            print(
                f"ATENÇÃO: valor_troca possui "
                f"{n_troca_nao_zero:,} valores diferentes de zero. Coluna mantida."
            )

    else:
        print("Coluna valor_troca não encontrada na base.")

    
    # 7. Criar flag de inconsistência financeira
    
    valor_liquido_calculado = (
        df["valor_bruto_venda"] - df["valor_desconto"]
    )

    df["flag_inconsistencia_financeira"] = (
        (df["faturamento_liquido"] - valor_liquido_calculado).abs() > 0.01
    ).astype(int)

    print(
        f"Inconsistências financeiras sinalizadas: "
        f"{df['flag_inconsistencia_financeira'].sum():,} "
        f"({df['flag_inconsistencia_financeira'].mean() * 100:.2f}%)"
    )

    
    # 8. Criar flag de devoluções / estornos
    
    df["flag_devolucao"] = (
        df["faturamento_liquido"] < 0
    ).astype(int)

    print(
        f"Registros negativos sinalizados: "
        f"{df['flag_devolucao'].sum():,} "
        f"({df['flag_devolucao'].mean() * 100:.2f}%)"
    )

    
    # 9. Criar flag de vendas de alto valor por loja
    
    p99_por_loja = (
        df[df["faturamento_liquido"] > 0]
        .groupby("loja")["faturamento_liquido"]
        .quantile(0.99)
    )

    df["limiar_p99_loja"] = df["loja"].map(p99_por_loja)

    df["flag_alto_valor"] = (
        (df["faturamento_liquido"] > df["limiar_p99_loja"])
        & (df["faturamento_liquido"] > 0)
    ).astype(int)

    print(
        f"Vendas de alto valor sinalizadas: "
        f"{df['flag_alto_valor'].sum():,} "
        f"({df['flag_alto_valor'].mean() * 100:.2f}%)"
    )

    
    # 10. Criar atributos temporais
    
    df["mes_numero"] = df["data_venda"].dt.month
    df["trimestre"] = df["data_venda"].dt.quarter
    df["mes_referencia"] = df["data_venda"].dt.to_period("M")

    
    # 11. Ordenar a base final
    
    df = (
        df.sort_values(["loja", "data_venda"])
        .reset_index(drop=True)
    )

    print(f"[OUT] Dimensão tratada: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

    print("=" * 80)
    print("FIM DO TRATAMENTO TRANSACIONAL")
    print("=" * 80)

    return df

## 5. Aplicação do Tratamento

Nesta etapa, a função de tratamento é aplicada sobre a base bruta consolidada.

In [9]:
#Definindo o df_tratado
df_tratado = tratamento_dados_transacionais(df)

display(df_tratado.head())

[IN] Dimensão original: 8,297,456 linhas × 14 colunas
Datas nulas removidas: 0
Outubro/2023 removido: 7,919 linhas
Possíveis erros de PDV removidos: 4
Código de produto preenchido: 3 registros
Coluna valor_troca removida: todos os valores eram iguais a zero.
Inconsistências financeiras sinalizadas: 983,249 (11.86%)
Registros negativos sinalizados: 374,570 (4.52%)
Vendas de alto valor sinalizadas: 68,031 (0.82%)
[OUT] Dimensão tratada: 8,289,533 linhas × 20 colunas
FIM DO TRATAMENTO TRANSACIONAL


,id_transacao,codigo_boleto,codigo_unidade,codigo_funcionario,data_venda,codigo_produto,codigo_categoria,qtd_boletos,qtd_produtos,valor_bruto_venda,valor_desconto,faturamento_liquido,loja,flag_inconsistencia_financeira,flag_devolucao,limiar_p99_loja,flag_alto_valor,mes_numero,trimestre,mes_referencia
0,E2B335CC-0666-4FDE-99F4-522CCF3F05B5,599560,A01B0C0F-788B-4AD2-B85C-89A48861AAD4,DBBB5204-138F-4B34-B937-0A977E81E25A,2022-01-02,C009D2EB-C185-4CF7-8F81-EDD468010C3A,43F98871-404E-4D3D-A810-B3EDF580E5E8,1,1.00,189.99,0.00,189.99,CASTING_DB063,0,0,449.99,0,1,1,2022-01
1,4443AFB1-A705-4BE0-B749-CB3659B47375,599561,A01B0C0F-788B-4AD2-B85C-89A48861AAD4,BCC8D5CE-660E-4A2E-B827-3FFA8F9E4499,2022-01-02,2F9B7B94-D775-4B00-8978-150E3990CCFD,C179DA0E-8301-4088-8373-E26B6FEE99C3,0,1.00,0.00,0.00,0.00,CASTING_DB063,0,0,449.99,0,1,1,2022-01
2,2601E03B-CC9B-43A8-807C-1896813DD246,599561,A01B0C0F-788B-4AD2-B85C-89A48861AAD4,BCC8D5CE-660E-4A2E-B827-3FFA8F9E4499,2022-01-02,A0BE1E55-782C-40D1-BC89-90AC605A245D,C179DA0E-8301-4088-8373-E26B6FEE99C3,1,1.00,0.00,0.00,0.00,CASTING_DB063,0,0,449.99,0,1,1,2022-01
3,04DB57E2-7AFD-4E36-BF34-97BF9145BE2C,599562,A01B0C0F-788B-4AD2-B85C-89A48861AAD4,9101091F-6771-4E60-B8CF-68121AC4BBC9,2022-01-02,DC9AB3D8-2239-4558-A836-58878975211B,D8BB8C86-1982-4D71-A085-80A75C213765,1,1.00,20.00,0.00,20.00,CASTING_DB063,0,0,449.99,0,1,1,2022-01
4,80613BDF-80D9-49F9-BE9C-AFE08D5E689E,599563,65BD060B-04F2-4D96-8ED7-F363F5C46874,EC1E9223-50F8-46D3-935B-DCE763CE89F1,2022-01-02,07665C97-34FD-4B0C-88F3-BFB72257A4E0,43F98871-404E-4D3D-A810-B3EDF580E5E8,1,1.00,14.17,0.00,14.17,CASTING_DB063,0,0,449.99,0,1,1,2022-01


## 6. Validação da Base Tratada

> Após aplicar o tratamento, é importante validar se as decisões foram executadas corretamente.

> Nesta etapa verificamos:

- período final da base;
- quantidade de lojas;
- colunas criadas;
- flags principais;
- ausência de outubro de 2023;
- remoção dos possíveis erros de PDV.

In [10]:
print(f'Dimensão da base tratada: {df_tratado.shape[0]:,} linhas × {df_tratado.shape[1]} colunas')
print(f'Período: {df_tratado["data_venda"].min().date()} até {df_tratado["data_venda"].max().date()}')
print(f'Lojas: {df_tratado["loja"].nunique()}')
print(f'Meses completos: {df_tratado["mes_referencia"].nunique()}')

Dimensão da base tratada: 8,289,533 linhas × 20 colunas
Período: 2022-01-01 até 2023-09-30
Lojas: 8
Meses completos: 21


In [11]:
# Validando se outubro/2023 foi removido
df_tratado['mes_referencia'].value_counts().sort_index().tail()

mes_referencia
2023-05    343466
2023-06    391723
2023-07    402051
2023-08    403322
2023-09    362342
Freq: M, Name: count, dtype: int64

In [12]:
# Validando se ainda existem erros de PDV após o tratamento
mask_erro_pdv_pos_tratamento = (
    (df_tratado['valor_desconto'] > df_tratado['valor_bruto_venda']) &
    (df_tratado['valor_bruto_venda'] > 0)
)

print(f'Possíveis erros de PDV após tratamento: {mask_erro_pdv_pos_tratamento.sum()}')

Possíveis erros de PDV após tratamento: 0


In [13]:
# Resumo das flags criadas
resumo_flags = pd.DataFrame({
    'flag': [
        'flag_inconsistencia_financeira',
        'flag_devolucao',
        'flag_alto_valor'
    ],
    'qtd_registros': [
        df_tratado['flag_inconsistencia_financeira'].sum(),
        df_tratado['flag_devolucao'].sum(),
        df_tratado['flag_alto_valor'].sum()
    ],
    'pct_base': [
        df_tratado['flag_inconsistencia_financeira'].mean() * 100,
        df_tratado['flag_devolucao'].mean() * 100,
        df_tratado['flag_alto_valor'].mean() * 100
    ]
})

resumo_flags['pct_base'] = resumo_flags['pct_base'].round(2)

display(resumo_flags)

,flag,qtd_registros,pct_base
0,flag_inconsistencia_financeira,983249,11.86
1,flag_devolucao,374570,4.52
2,flag_alto_valor,68031,0.82


In [14]:
# Resumo por loja das principais flags
resumo_flags_loja = (
    df_tratado
    .groupby('loja')
    .agg(
        qtd_registros=('loja', 'size'),
        qtd_inconsist_financeira=('flag_inconsistencia_financeira', 'sum'),
        qtd_devolucoes=('flag_devolucao', 'sum'),
        qtd_alto_valor=('flag_alto_valor', 'sum')
    )
)

resumo_flags_loja['pct_inconsist_financeira'] = (
    resumo_flags_loja['qtd_inconsist_financeira'] /
    resumo_flags_loja['qtd_registros'] * 100
).round(2)

resumo_flags_loja['pct_devolucoes'] = (
    resumo_flags_loja['qtd_devolucoes'] /
    resumo_flags_loja['qtd_registros'] * 100
).round(2)

resumo_flags_loja['pct_alto_valor'] = (
    resumo_flags_loja['qtd_alto_valor'] /
    resumo_flags_loja['qtd_registros'] * 100
).round(2)

display(resumo_flags_loja)

,qtd_registros,qtd_inconsist_financeira,qtd_devolucoes,qtd_alto_valor,pct_inconsist_financeira,pct_devolucoes,pct_alto_valor
loja,,,,,,,
CASTING_DB063,2853694,22,8533,21255,0.00,0.30,0.74
CASTING_DB069,93551,0,2743,907,0.00,2.93,0.97
CASTING_DB088,274937,4090,4090,2586,1.49,1.49,0.94
CASTING_DB092,786938,20827,20827,7627,2.65,2.65,0.97
CASTING_DB101,223464,2,14340,2092,0.00,6.42,0.94
CASTING_DB102,283595,0,12349,2713,0.00,4.35,0.96
CASTING_DB109,1391360,18001,91007,9256,1.29,6.54,0.67
CASTING_DB113,2381994,940307,220681,21595,39.48,9.26,0.91


## 7. Agregação Mensal para Modelagem

> Como o problema será tratado como previsão de faturamento em série temporal, a base precisa ser agregada em granularidade mensal.

> A agregação será feita por:

- mês;
- loja.

> A variável-alvo da modelagem será:

> `faturamento_liquido_mensal`

> Essa variável representa a soma mensal de `faturamento_liquido`.

In [15]:
def agregar_mensal(df_tratado: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega a base transacional tratada em granularidade mensal por loja.

    A base resultante será utilizada como entrada para a etapa de modelagem.

    Observação:
    Algumas variáveis agregadas são úteis para análise histórica e diagnóstico.
    Porém, na modelagem preditiva, deve-se tomar cuidado para não usar variáveis
    que não estarão disponíveis no momento da previsão futura.
    """

    agg = (
        df_tratado
        .groupby(['mes_referencia', 'loja'])
        .agg(
            faturamento_liquido_mensal=('faturamento_liquido', 'sum'),
            FAT_BRUTO_POS_MES=('valor_bruto_venda', lambda x: x[x > 0].sum()),
            TOTAL_DESCONTOS_POS=('valor_desconto', lambda x: x[x > 0].sum()),
            TOTAL_REG_NEG_VALOR=('faturamento_liquido', lambda x: x[x < 0].sum()),
            QTD_TRANSACOES=('id_transacao', 'count'),
            QTD_REG_NEG=('flag_devolucao', 'sum'),
            QTD_ALTO_VALOR=('flag_alto_valor', 'sum'),
            QTD_INCONSIST_FINANCEIRA=('flag_inconsistencia_financeira', 'sum'),
            TICKET_MEDIO_POS=('faturamento_liquido', lambda x: x[x > 0].mean()),
            TICKET_MEDIano_POS=('faturamento_liquido', lambda x: x[x > 0].median())
        )
        .reset_index()
    )

    # Taxa de registros negativos em valor
    agg['TAXA_REG_NEG_PCT'] = np.where(
        agg['FAT_BRUTO_POS_MES'] > 0,
        agg['TOTAL_REG_NEG_VALOR'].abs() / agg['FAT_BRUTO_POS_MES'] * 100,
        0
    )

    # Taxa de desconto sobre faturamento bruto positivo
    agg['TAXA_DESCONTO_PCT'] = np.where(
        agg['FAT_BRUTO_POS_MES'] > 0,
        agg['TOTAL_DESCONTOS_POS'] / agg['FAT_BRUTO_POS_MES'] * 100,
        0
    )

    # Percentual de registros negativos em quantidade
    agg['PCT_QTD_REG_NEG'] = np.where(
        agg['QTD_TRANSACOES'] > 0,
        agg['QTD_REG_NEG'] / agg['QTD_TRANSACOES'] * 100,
        0
    )

    # Percentual de vendas de alto valor em quantidade
    agg['PCT_QTD_ALTO_VALOR'] = np.where(
        agg['QTD_TRANSACOES'] > 0,
        agg['QTD_ALTO_VALOR'] / agg['QTD_TRANSACOES'] * 100,
        0
    )

    # Percentual de inconsistências financeiras
    agg['PCT_INCONSIST_FINANCEIRA'] = np.where(
        agg['QTD_TRANSACOES'] > 0,
        agg['QTD_INCONSIST_FINANCEIRA'] / agg['QTD_TRANSACOES'] * 100,
        0
    )

    # Ordenação final
    agg = agg.sort_values(['loja', 'mes_referencia']).reset_index(drop=True)

    return agg

## 8. Validação da Base Mensal

> Após a agregação mensal, verificamos se a base final está coerente com o escopo da modelagem.

> Esperamos obter:

- 8 lojas;
- 21 meses completos;
- 168 observações, considerando 8 lojas × 21 meses.

In [16]:
df_mensal = agregar_mensal(df_tratado)

print(f'Dimensão da base mensal: {df_mensal.shape[0]} linhas × {df_mensal.shape[1]} colunas')
print(f'Lojas: {df_mensal["loja"].nunique()}')
print(f'Períodos: {df_mensal["mes_referencia"].nunique()}')
print(f'Período inicial: {df_mensal["mes_referencia"].min()}')
print(f'Período final: {df_mensal["mes_referencia"].max()}')

display(df_mensal.head(10))

Dimensão da base mensal: 168 linhas × 17 colunas
Lojas: 8
Períodos: 21
Período inicial: 2022-01
Período final: 2023-09


,mes_referencia,loja,faturamento_liquido_mensal,FAT_BRUTO_POS_MES,TOTAL_DESCONTOS_POS,TOTAL_REG_NEG_VALOR,QTD_TRANSACOES,QTD_REG_NEG,QTD_ALTO_VALOR,QTD_INCONSIST_FINANCEIRA,TICKET_MEDIO_POS,TICKET_MEDIano_POS,TAXA_REG_NEG_PCT,TAXA_DESCONTO_PCT,PCT_QTD_REG_NEG,PCT_QTD_ALTO_VALOR,PCT_INCONSIST_FINANCEIRA
0,2022-01,CASTING_DB063,10359240.33,10412906.72,14523.96,-39142.43,111828,318,298,0,114.06,99.99,0.38,0.14,0.28,0.27,0.00
1,2022-02,CASTING_DB063,10980198.36,11033972.01,19952.95,-33820.70,107689,285,409,0,116.06,99.99,0.31,0.18,0.26,0.38,0.00
2,2022-03,CASTING_DB063,13205997.56,13275769.21,21261.20,-48510.45,122772,422,529,0,122.11,99.99,0.37,0.16,0.34,0.43,0.00
3,2022-04,CASTING_DB063,16111945.79,16206681.09,29943.10,-64905.59,129551,424,885,3,140.17,129.99,0.40,0.18,0.33,0.68,0.00
4,2022-05,CASTING_DB063,22810386.07,22925904.39,43402.29,-72116.05,170512,437,1447,0,152.78,149.99,0.31,0.19,0.26,0.85,0.00
5,2022-06,CASTING_DB063,20468268.66,20603058.02,46868.46,-87923.90,150598,534,1553,2,156.27,149.99,0.43,0.23,0.35,1.03,0.00
6,2022-07,CASTING_DB063,15679438.79,15813916.14,31100.86,-103376.49,124029,711,802,0,143.82,137.20,0.65,0.20,0.57,0.65,0.00
7,2022-08,CASTING_DB063,16309237.28,16398593.59,39160.01,-50196.30,126824,356,840,0,147.15,139.99,0.31,0.24,0.28,0.66,0.00
8,2022-09,CASTING_DB063,15128928.49,15206287.24,33660.05,-43698.70,114218,336,729,0,149.54,149.99,0.29,0.22,0.29,0.64,0.00
9,2022-10,CASTING_DB063,17460713.27,17540602.15,32262.04,-47626.84,138177,360,709,0,143.08,129.99,0.27,0.18,0.26,0.51,0.00


In [17]:
# Verificar se todas as lojas possuem todos os meses
validacao_meses_loja = (
    df_mensal
    .groupby('loja')['mes_referencia']
    .nunique()
    .sort_values()
)

display(validacao_meses_loja.to_frame('qtd_meses'))

,qtd_meses
loja,
CASTING_DB063,21
CASTING_DB069,21
CASTING_DB088,21
CASTING_DB092,21
CASTING_DB101,21
CASTING_DB102,21
CASTING_DB109,21
CASTING_DB113,21


In [18]:
# Validação do faturamento mensal consolidado
fat_total_transacional = df_tratado['faturamento_liquido'].sum()
fat_total_mensal = df_mensal['faturamento_liquido_mensal'].sum()

print(f'Faturamento total na base tratada : R$ {fat_total_transacional:,.2f}')
print(f'Faturamento total na base mensal  : R$ {fat_total_mensal:,.2f}')
print(f'Diferença                         : R$ {fat_total_transacional - fat_total_mensal:,.2f}')

Faturamento total na base tratada : R$ 1,893,542,828.94
Faturamento total na base mensal  : R$ 1,893,542,828.94
Diferença                         : R$ -0.00


In [19]:
# Visualização simples da base mensal por loja
display(
    df_mensal
    .groupby('loja')
    .agg(
        fat_liquido_total=('faturamento_liquido_mensal', 'sum'),
        fat_liquido_medio_mensal=('faturamento_liquido_mensal', 'mean'),
        qtd_meses=('mes_referencia', 'nunique')
    )
    .sort_values('fat_liquido_total', ascending=False)
)

,fat_liquido_total,fat_liquido_medio_mensal,qtd_meses
loja,,,
CASTING_DB113,650267356.19,30965112.20,21
CASTING_DB109,382271717.24,18203415.11,21
CASTING_DB063,356879171.80,16994246.28,21
CASTING_DB088,153528645.37,7310887.87,21
CASTING_DB102,128475430.84,6117877.66,21
CASTING_DB101,103266167.94,4917436.57,21
CASTING_DB092,80147921.17,3816567.67,21
CASTING_DB069,38706418.39,1843162.78,21


## 9. Exportação dos Arquivos Tratados

Nesta etapa, os arquivos finais são salvos na pasta `data/processed`.

Serão gerados dois arquivos principais:

- `df_tratado.csv`: base transacional tratada;
- `df_mensal.csv`: base mensal por loja pronta para modelagem.

In [20]:
# Criar pasta processed, caso ainda não exista
caminho_pasta = '../data/processed'

os.makedirs(caminho_pasta, exist_ok=True)

# Caminhos dos arquivos finais
caminho_df_tratado = f'{caminho_pasta}/df_tratado.csv'
caminho_df_mensal = f'{caminho_pasta}/df_mensal.csv'

# Exportar arquivos
df_tratado.to_csv(caminho_df_tratado, index=False)
df_mensal.to_csv(caminho_df_mensal, index=False)

print('Arquivos salvos com sucesso!')
print(f'Base transacional tratada: {caminho_df_tratado}')
print(f'Base mensal para modelagem: {caminho_df_mensal}')

Arquivos salvos com sucesso!
Base transacional tratada: ../data/processed/df_tratado.csv
Base mensal para modelagem: ../data/processed/df_mensal.csv


## 10. Conclusão do Tratamento

> O tratamento de dados teve como objetivo transformar a base transacional bruta em um conjunto de dados consistente e adequado para a modelagem preditiva de faturamento.

> As decisões aplicadas foram fundamentadas na etapa de Análise Exploratória de Dados (EDA). O mês de outubro de 2023 foi removido por representar um período incompleto, evitando vieses na série temporal. Registros com possíveis erros de operação (`valor_desconto` superior ao `valor_bruto_venda` em vendas positivas) foram excluídos, enquanto devoluções e estornos foram preservados por refletirem eventos reais que impactam o `faturamento_liquido`.

> A coluna `valor_troca` foi removida por não apresentar variabilidade durante todo o período analisado, não agregando informação ao modelo. Já as diferenças entre `valor_bruto_venda - valor_desconto` e `faturamento_liquido` foram tratadas por meio da criação de uma flag de inconsistência financeira. Como a exploração dos dados indicou que essas divergências fazem parte do processo operacional e não podem ser totalmente explicadas pelas variáveis disponíveis, optou-se por manter `faturamento_liquido` como referência oficial da variável-alvo.

> Além disso, foram criadas variáveis auxiliares para enriquecer a base de dados, incluindo indicadores de devolução, vendas de alto valor por loja e atributos temporais. Essas informações permitem identificar comportamentos relevantes sem eliminar registros que podem conter padrões importantes para a aprendizagem do modelo.

> Como resultado do processo de tratamento:

- foram removidos 7.919 registros referentes ao mês incompleto de outubro de 2023;
- apenas 4 registros foram identificados como possíveis erros operacionais e excluídos;
- 983.249 transações (11,86%) foram sinalizadas com inconsistência financeira para rastreabilidade;
- 374.570 registros (4,52%) foram classificados como devoluções ou estornos;
- 68.031 transações (0,82%) foram identificadas como vendas de alto valor considerando o percentil 99 de cada loja.

> Ao final, foram geradas duas bases para as próximas etapas do projeto:

- `df_tratado.csv`: base transacional tratada, contendo os registros preparados para análises detalhadas;
- `df_mensal.csv`: base agregada mensalmente por loja, utilizada na construção e avaliação dos modelos de Machine Learning.

> Essas etapas garantem maior qualidade, consistência e rastreabilidade dos dados, reduzindo a influência de inconsistências operacionais e preservando eventos reais do negócio. Dessa forma, a base está preparada para a etapa de engenharia de atributos e desenvolvimento do modelo preditivo de faturamento.

> No próximo notebook, a base tratada será utilizada para a construção das variáveis temporais, treinamento dos modelos de Machine Learning e geração das previsões de faturamento.